# How X picked 1 of 4 notes
One tweet ("SAUDI IS BURNING") got 4 Community Notes. Only Note 4 was shown. This notebook follows the same steps as the diagram:
notes → ratings → raters → trusted or not → which model → X's scores.

In [1]:
import glob

import pandas as pd

# ids are 19 digits, so read them as text (otherwise pandas rounds them)
id_columns = {"noteId": str, "tweetId": str, "noteAuthorParticipantId": str, "raterParticipantId": str, "participantId": str}

saudi_folder = "dataset/tweet_2109031558967615703/"
combined = pd.read_csv(saudi_folder + "combined.csv", dtype=id_columns)
notes = pd.read_csv(saudi_folder + "notes.csv", dtype=id_columns)
note_status = pd.read_csv(saudi_folder + "note_status.csv", dtype=id_columns)
ratings = pd.read_csv(saudi_folder + "ratings.csv", dtype=id_columns)

note_names = ["Note 1", "Note 2", "Note 3", "Note 4"]
print("tweet:", combined["tweet_text"][0])
print("notes:", len(notes), "| ratings:", len(ratings), "| raters:", ratings["raterParticipantId"].nunique())

tweet: SAUDI IS BURNING
notes: 4 | ratings: 456 | raters: 263


## 1. The 4 notes
`writer_type` comes from the writer's status in X's user file: `apiEarnedIn` means an AI note writer.

In [2]:
tweet_time = pd.to_datetime(combined["tweet_created_at"][0], utc=True)
notes["minutes_after_tweet"] = (pd.to_datetime(notes["createdAtMillis"], unit="ms", utc=True) - tweet_time).dt.total_seconds() // 60
notes = notes.merge(note_status[["noteId", "currentStatus"]], on="noteId")

notes[["note_label", "writer_type", "writer_enrollmentState", "minutes_after_tweet", "currentStatus", "summary"]]

,note_label,writer_type,writer_enrollmentState,minutes_after_tweet,currentStatus,summary
0,Note 1,Human,earnedIn,15.0,NEEDS_MORE_RATINGS,Mario claims saudi arabia is burning after an ...
1,Note 2,AI,apiEarnedIn,48.0,NEEDS_MORE_RATINGS,Saudi Arabia faces targeted Houthi attacks on ...
2,Note 3,AI,apiEarnedIn,118.0,NEEDS_MORE_RATINGS,Saudi Arabia is not &quot;burning.&quot; Offic...
3,Note 4,Human,earnedIn,119.0,CURRENTLY_RATED_HELPFUL,“Saudi is burning” misleadingly implies nati...


In [3]:
# how active are these writers? count every note in X's data (takes about a minute)
all_notes = pd.concat(pd.read_csv(f, sep="\t", usecols=["noteAuthorParticipantId"], dtype=id_columns)
                      for f in sorted(glob.glob("data/notes/notes-*.tsv")))
users = pd.read_csv("data/userEnrollment-00000.tsv", sep="\t", usecols=["participantId", "enrollmentState"], dtype=id_columns)
ai_writers = set(users.loc[users["enrollmentState"] == "apiEarnedIn", "participantId"])

notes_per_writer = all_notes["noteAuthorParticipantId"].value_counts()
ai_ranking = list(notes_per_writer[notes_per_writer.index.isin(ai_writers)].index)

for _, note in notes.iterrows():
    writer = note["noteAuthorParticipantId"]
    if writer in ai_ranking:
        print(note["note_label"], "- AI, rank", ai_ranking.index(writer) + 1, "of", len(ai_ranking), "AI accounts,", notes_per_writer[writer], "notes")
    else:
        print(note["note_label"], "- human,", notes_per_writer[writer], "notes ever")

Note 1 - human, 3 notes ever
Note 2 - AI, rank 2 of 36 AI accounts, 45737 notes
Note 3 - AI, rank 1 of 36 AI accounts, 46308 notes
Note 4 - human, 1 notes ever


Notes 2 and 3 come from the 2 most active AI accounts on X. Note 4 was a person's first note.

## 2. The ratings

In [4]:
for name in note_names:
    levels = ratings.loc[ratings["note_label"] == name, "helpfulnessLevel"]
    print(name, "|", len(levels), "ratings |", round((levels == "HELPFUL").mean() * 100), "% Helpful")

Note 1 | 113 ratings | 77 % Helpful
Note 2 | 101 ratings | 76 % Helpful
Note 3 | 77 ratings | 83 % Helpful
Note 4 | 165 ratings | 82 % Helpful


All 4 notes got about 80% Helpful, so the vote count alone doesn't explain why only Note 4 was shown.

## 3. The raters
`rater_modelingPopulation`: CORE (older countries) or EXPANSION (newer countries). `rater_modelingGroup`: X's region group.

In [5]:
raters = ratings.drop_duplicates("raterParticipantId")
print("raters by population:", raters["rater_modelingPopulation"].value_counts().to_dict())
print("biggest groups:", raters["rater_modelingGroup"].value_counts().head(2).to_dict())
print()

# % Helpful from the 2 biggest groups
for name in note_names:
    this_note = ratings[ratings["note_label"] == name]
    group_24 = this_note[this_note["rater_modelingGroup"] == 24]
    group_13 = this_note[this_note["rater_modelingGroup"] == 13]
    print(name, "| group 24:", round((group_24["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful",
          "| group 13:", round((group_13["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful")

raters by population: {'EXPANSION': 204, 'CORE': 57, 'EXPANSION_PLUS': 2}
biggest groups: {24.0: 184, 13.0: 37}

Note 1 | group 24: 92 % Helpful | group 13: 29 % Helpful
Note 2 | group 24: 91 % Helpful | group 13: 53 % Helpful
Note 3 | group 24: 97 % Helpful | group 13: 44 % Helpful
Note 4 | group 24: 96 % Helpful | group 13: 45 % Helpful


Group 24 liked every note. Group 13 was split. So the two groups disagree.

## 4. Trusted or not trusted
X's rule (from `helpfulness_scores.py` in X's code):
- take the rater's past ratings that were Helpful or Not Helpful, made within 48 hours and before X decided
- trust score = ratings that matched X's final decision / those ratings
- trusted = trust score 0.66 or more and 10+ ratings

This reads all of X's ratings files, so it takes about 3 minutes.

In [6]:
# every rating our raters ever made
rater_ids = set(ratings["raterParticipantId"])
parts = []
for file in sorted(glob.glob("data/ratings/ratings-*.tsv")):
    for chunk in pd.read_csv(file, sep="\t", dtype=id_columns, chunksize=5_000_000,
                             usecols=["noteId", "raterParticipantId", "createdAtMillis", "helpfulnessLevel", "helpful", "notHelpful"]):
        parts.append(chunk[chunk["raterParticipantId"].isin(rater_ids)])
history = pd.concat(parts)

# add the final decision of each note they rated
status_file = pd.read_csv("data/noteStatusHistory-00000.tsv", sep="\t", dtype=id_columns,
                          usecols=["noteId", "createdAtMillis", "currentStatus", "timestampMillisOfFirstNonNMRStatus"])
history = history.merge(status_file.rename(columns={"createdAtMillis": "note_created_ms"}), on="noteId", how="left")
print("past ratings of our raters:", len(history))

past ratings of our raters: 218395


In [7]:
# 1 = Helpful, 0 = Not Helpful (old ratings use the helpful / notHelpful columns)
history["value"] = float("nan")
history.loc[(history["helpful"] == 1) | (history["helpfulnessLevel"] == "HELPFUL"), "value"] = 1
history.loc[(history["notHelpful"] == 1) | (history["helpfulnessLevel"] == "NOT_HELPFUL"), "value"] = 0

# valid ratings: X's 4 conditions
note_decided = history["currentStatus"].isin(["CURRENTLY_RATED_HELPFUL", "CURRENTLY_RATED_NOT_HELPFUL"])
within_48_hours = history["createdAtMillis"] - history["note_created_ms"] < 48 * 60 * 60 * 1000
before_decision = history["createdAtMillis"] < history["timestampMillisOfFirstNonNMRStatus"]
helpful_or_not = history["value"].isin([0, 1])
valid = history[note_decided & within_48_hours & before_decision & helpful_or_not].copy()

# did the rating match X's final decision?
valid["matched"] = ((valid["value"] == 1) & (valid["currentStatus"] == "CURRENTLY_RATED_HELPFUL")) | \
                   ((valid["value"] == 0) & (valid["currentStatus"] == "CURRENTLY_RATED_NOT_HELPFUL"))
print("valid ratings:", len(valid), "| matched:", valid["matched"].sum())

valid ratings: 31916 | matched: 27760


In [8]:
# one row per rater
profile = raters.set_index("raterParticipantId")[["rater_modelingPopulation", "rater_modelingGroup"]].copy()
profile["total_ratings"] = history.groupby("raterParticipantId").size()
profile["valid_ratings"] = valid.groupby("raterParticipantId").size()
profile["agreeing_ratings"] = valid.groupby("raterParticipantId")["matched"].sum()
profile = profile.fillna({"valid_ratings": 0, "agreeing_ratings": 0})

profile["trust_score"] = (profile["agreeing_ratings"] / profile["valid_ratings"]).round(2)
profile["trusted"] = (profile["total_ratings"] >= 10) & (profile["trust_score"] >= 0.66)
profile.to_csv(saudi_folder + "rater_profiles.csv")

print("trusted:", profile["trusted"].sum(), "| not trusted:", (~profile["trusted"]).sum())
profile.sample(5, random_state=1)

trusted: 227 | not trusted: 36


,rater_modelingPopulation,rater_modelingGroup,total_ratings,valid_ratings,agreeing_ratings,trust_score,trusted
raterParticipantId,,,,,,,
476ED34259534E557194F17F91E36217B5DEA39BD13F7C8B332DF06BCB80B479,EXPANSION,24.0,107,12.0,11.0,0.92,True
A59037968AA597CFFE41F43BF66785782197393F9053EE25999545B6EEC24AC5,CORE,13.0,1240,152.0,111.0,0.73,True
C8998F4E47BE5D5771CFE2F1DE41D89A6235A56BE1FEF74D6DCDDEAF02DB4406,EXPANSION,24.0,26,1.0,1.0,1.00,True
E21906CD42CD29031B5DAF1BF5A0BD052B43F52157EB324D06DFB302FCB32518,EXPANSION,27.0,43,8.0,8.0,1.00,True
38803E6524398B188C6CA75065532A23784FC2E708FD66FB7F177BF79725E2B8,EXPANSION,24.0,23,4.0,4.0,1.00,True


227 of 263 raters are trusted. Saved in `rater_profiles.csv` (column `trusted`).

## 5. Which model decides
X has a Core model and an Expansion model. A model only decides a note if half its ratings come from its own groups (`scorer.py`).

In [9]:
# group lists from X's code (constants.py)
core_groups = {1, 2, 3, 6, 8, 9, 10, 11, 13, 14, 19, 21, 25}

for name in note_names:
    groups = ratings.loc[ratings["note_label"] == name, "rater_modelingGroup"]
    core_share = groups.isin(core_groups).mean() * 100
    if core_share >= 50:
        model = "Core"
    else:
        model = "Expansion"
    print(name, "| ratings from Core groups:", round(core_share), "% | decided by:", model)

print("X's data says:", note_status["currentDecidedBy"].unique())

Note 1 | ratings from Core groups: 22 % | decided by: Expansion
Note 2 | ratings from Core groups: 25 % | decided by: Expansion
Note 3 | ratings from Core groups: 21 % | decided by: Expansion
Note 4 | ratings from Core groups: 19 % | decided by: Expansion
X's data says: <StringArray>
['ExpansionModel (v1.1)']
Length: 1, dtype: str


Only 19-25% of ratings came from Core groups, so the Expansion model decided. Same as X's data.

## 6. X's scores: why Note 4 won
`score_with_x_model.py` runs X's own model on one month of ratings and saves `model_scores.csv` and `rater_sides.csv`.
Each note gets a helpfulness score and a one-sidedness number. X blocks notes with one-sidedness 0.5 or more.

In [10]:
model_scores = pd.read_csv(saudi_folder + "model_scores.csv", dtype=id_columns)
model_scores["one_sided"] = model_scores["one_sidedness"].abs() >= 0.5
model_scores[["note_label", "writer_type", "helpfulness_score", "one_sidedness", "one_sided", "currentStatus"]].round(2)

,note_label,writer_type,helpfulness_score,one_sidedness,one_sided,currentStatus
0,Note 1,Human,0.12,0.57,True,NEEDS_MORE_RATINGS
1,Note 2,AI,0.15,0.56,True,NEEDS_MORE_RATINGS
2,Note 3,AI,0.15,0.57,True,NEEDS_MORE_RATINGS
3,Note 4,Human,0.21,0.32,False,CURRENTLY_RATED_HELPFUL


In [11]:
# how each side rated the notes, before Note 4 was shown
sides = pd.read_csv(saudi_folder + "rater_sides.csv", dtype=id_columns).set_index("raterParticipantId")
ratings["rater_side"] = ratings["raterParticipantId"].map(sides["rater_side"])
note4_shown_ms = note_status.loc[note_status["note_label"] == "Note 4", "timestampMillisOfFirstNonNMRStatus"].iloc[0]
before_note4 = ratings[(ratings["createdAtMillis"] <= note4_shown_ms) & ratings["rater_side"].notna()]

for name in note_names:
    this_note = before_note4[before_note4["note_label"] == name]
    main_side = this_note[this_note["rater_side"] > 0]
    other_side = this_note[this_note["rater_side"] <= 0]
    print(name, "| main side:", round((main_side["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful",
          "| other side:", round((other_side["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful")

Note 1 | main side: 84 % Helpful | other side: 31 % Helpful
Note 2 | main side: 87 % Helpful | other side: 36 % Helpful
Note 3 | main side: 90 % Helpful | other side: 40 % Helpful
Note 4 | main side: 86 % Helpful | other side: 47 % Helpful


Notes 1-3 are one-sided. Note 4 is not, and it has the highest score. The other side gave Note 4 47% Helpful (others 31-40%),
so Note 4 was the only note liked by both sides.

Note: my scores use one month of data, so they are lower than X's. Compare the notes with each other.